# Cache Invalidation [Step 2 - When to Invalidate or Update Cache]

> **MLCourse - Agentic AI - Cache RAG**

Caching without invalidation returns stale answers. This notebook builds
three invalidation strategies: time-to-live (TTL), content-based detection,
and manual invalidation. We demonstrate when cache entries become stale
and how each strategy catches or prevents serving outdated information.

In [1]:
import os
import time
import hashlib
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


### 1. Load and Chunk the Document


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print(f"Loaded alice.txt: {len(raw_text)} chars -> {len(chunks)} chunks")


### 2. Build the Vector Store


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_invalidation")
print(f"Vector store built with {vectorstore._collection.count()} vectors")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


### 3. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)


### 4. Define the Three Invalidation Strategies


In [ ]:
# ### Strategy A: Time-To-Live (TTL)
# Each cache entry has a maximum age. Once exceeded, the entry is
# considered stale and must be regenerated. Simple but effective:
# guarantees freshness at the cost of re-computing valid answers.

class TTLCache:
    """Cache with time-to-live invalidation."""

    def __init__(self, ttl_seconds=300):
        self.ttl = ttl_seconds
        self.cache = {}  # key -> {answer, timestamp}

    def _make_key(self, query):
        return query.strip().lower()

    def get(self, query):
        key = self._make_key(query)
        if key not in self.cache:
            return None, "miss"
        entry = self.cache[key]
        age = time.time() - entry["timestamp"]
        if age > self.ttl:
            return None, "expired"
        return entry["answer"], "hit"

    def put(self, query, answer):
        key = self._make_key(query)
        self.cache[key] = {
            "answer": answer,
            "timestamp": time.time(),
        }

    def invalidate_all(self):
        self.cache.clear()

    def size(self):
        return len(self.cache)

    def inspect(self):
        results = []
        now = time.time()
        for key, entry in self.cache.items():
            age = now - entry["timestamp"]
            stale = age > self.ttl
            results.append({
                "key": key,
                "age_seconds": round(age, 1),
                "stale": stale,
                "ttl": self.ttl,
            })
        return results

print("TTLCache defined (default TTL: 300 seconds)")

# ### Strategy B: Content-Based Invalidation
# We compute a hash of the source document. If the document changes,
# all cache entries become stale. This is useful when the knowledge
# base is updated and cached answers reference old content.

class ContentHashCache:
    """Cache invalidated when source content hash changes."""

    def __init__(self):
        self.cache = {}
        self.content_hash = None  # hash of the source document

    def _make_key(self, query):
        return query.strip().lower()

    def set_content_hash(self, text):
        """Set the hash of the current source document."""
        self.content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

    def get(self, query, current_hash=None):
        key = self._make_key(query)
        if key not in self.cache:
            return None, "miss"
        entry = self.cache[key]
        if current_hash and current_hash != entry.get("source_hash"):
            return None, "content_changed"
        return entry["answer"], "hit"

    def put(self, query, answer):
        key = self._make_key(query)
        self.cache[key] = {
            "answer": answer,
            "source_hash": self.content_hash,
        }

    def invalidate_all(self):
        self.cache.clear()

    def size(self):
        return len(self.cache)

print("ContentHashCache defined")

# ### Strategy C: Manual Invalidation
# Provides explicit API to invalidate specific queries or all entries.
# Useful for user-driven corrections or admin actions.

class ManualCache:
    """Cache with explicit manual invalidation API."""

    def __init__(self):
        self.cache = {}
        self.invalidated = set()  # track invalidated keys

    def _make_key(self, query):
        return query.strip().lower()

    def get(self, query):
        key = self._make_key(query)
        if key in self.invalidated:
            return None, "manually_invalidated"
        if key not in self.cache:
            return None, "miss"
        return self.cache[key]["answer"], "hit"

    def put(self, query, answer):
        key = self._make_key(query)
        self.cache[key] = {"answer": answer}
        self.invalidated.discard(key)  # remove from invalidated if re-stored

    def invalidate(self, query):
        """Invalidate a specific query."""
        key = self._make_key(query)
        self.invalidated.add(key)

    def invalidate_all(self):
        self.cache.clear()
        self.invalidated.clear()

    def size(self):
        return len(self.cache)

print("ManualCache defined")


### 5. Initialize All Three Caches


In [ ]:
ttl_cache = TTLCache(ttl_seconds=5)  # short TTL for demo
content_cache = ContentHashCache()
manual_cache = ManualCache()

# Set the content hash for the current document
content_cache.set_content_hash(raw_text)

print(f"TTL cache: TTL={ttl_cache.ttl}s")
print(f"Content cache: hash={content_cache.content_hash}")
print("Manual cache: ready")


### 6. Helper: Generate Answer (Simulates RAG)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question using ONLY the provided context. "
     "Be concise and accurate."),
    ("user", "Context:\n{context}\n\nQuestion: {query}")
])

def generate_answer(query):
    """Simulate a RAG generation step."""
    docs = retriever.invoke(query)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    response = (rag_prompt | llm).invoke({"context": context, "query": query})
    return response.content

print("generate_answer function defined.")


### 7. Strategy A Demo: TTL Invalidation


In [ ]:
print("=" * 60)
print("STRATEGY A: TTL-BASED INVALIDATION")
print("=" * 60)

# Populate cache
query = "Who does Alice meet at the tea party?"
answer = generate_answer(query)
ttl_cache.put(query, answer)
print(f"\n[STORE] '{query[:50]}...'")
print(f"  Answer: {answer[:100]}...")

# Immediate lookup -- should hit
result, status = ttl_cache.get(query)
print(f"\n[LOOKUP] status={status}")
if result:
    print(f"  Answer: {result[:100]}...")

# Wait for TTL to expire
print(f"\nWaiting {ttl_cache.ttl + 1} seconds for TTL to expire...")
time.sleep(ttl_cache.ttl + 1)

# Lookup after expiry -- should be expired
result, status = ttl_cache.get(query)
print(f"\n[LOOKUP AFTER EXPIRY] status={status}")
if result is None:
    print("  Cache expired! Must regenerate.")

# Regenerate
new_answer = generate_answer(query)
ttl_cache.put(query, new_answer)
print(f"\n[REGENERATED] New answer stored.")
print(f"  Answer: {new_answer[:100]}...")

# Inspect TTL cache state
print("\nCache inspection:")
for entry in ttl_cache.inspect():
    print(f"  key: '{entry['key'][:40]}...' | age: {entry['age_seconds']}s | stale: {entry['stale']}")


### 8. Strategy B Demo: Content-Based Invalidation


In [ ]:
print("\n" + "=" * 60)
print("STRATEGY B: CONTENT-BASED INVALIDATION")
print("=" * 60)

# Populate cache
query_b = "What happens when Alice follows the White Rabbit?"
answer_b = generate_answer(query_b)
content_cache.put(query_b, answer_b)
print(f"\n[STORE] '{query_b[:50]}...'")
print(f"  Answer: {answer_b[:100]}...")

# Lookup with matching hash -- should hit
result, status = content_cache.get(query_b, current_hash=content_cache.content_hash)
print(f"\n[LOOKUP] status={status} (hash matches)")

# Simulate document change: modify the hash
modified_hash = "fake_hash_12345"
result, status = content_cache.get(query_b, current_hash=modified_hash)
print(f"\n[LOOKUP] status={status} (hash changed!)")
if result is None:
    print("  Content changed! Cache invalidated.")
    print("  Must regenerate from updated document.")

# Regenerate with new hash
content_cache.content_hash = modified_hash
new_answer_b = generate_answer(query_b)
content_cache.put(query_b, new_answer_b)
print(f"\n[REGENERATED] New answer stored with updated hash.")

# Lookup now hits again
result, status = content_cache.get(query_b, current_hash=modified_hash)
print(f"\n[LOOKUP] status={status} (hash matches again)")


### 9. Strategy C Demo: Manual Invalidation


In [ ]:
print("\n" + "=" * 60)
print("STRATEGY C: MANUAL INVALIDATION")
print("=" * 60)

# Populate cache
query_c = "Describe the Cheshire Cat"
answer_c = generate_answer(query_c)
manual_cache.put(query_c, answer_c)
print(f"\n[STORE] '{query_c[:50]}...'")
print(f"  Answer: {answer_c[:100]}...")

# Lookup -- should hit
result, status = manual_cache.get(query_c)
print(f"\n[LOOKUP] status={status}")

# Manually invalidate
manual_cache.invalidate(query_c)
print(f"\n[INVALIDATED] '{query_c[:50]}...'")

# Lookup after invalidation
result, status = manual_cache.get(query_c)
print(f"\n[LOOKUP] status={status}")
if result is None:
    print("  Entry invalidated. Must regenerate.")

# Regenerate and store
new_answer_c = generate_answer(query_c)
manual_cache.put(query_c, new_answer_c)
print(f"\n[REGENERATED] Answer stored again.")

# Invalidating everything
manual_cache.invalidate_all()
print(f"\n[INVALIDATE ALL] Cache cleared. Size: {manual_cache.size()}")


### 10. Staleness Detection Demo


In [ ]:
# Show how we detect stale entries before serving them.

print("\n" + "=" * 60)
print("STALENESS DETECTION")
print("=" * 60)

ttl_cache2 = TTLCache(ttl_seconds=3)
query_d = "What is the Mad Hatter's riddle?"
answer_d = generate_answer(query_d)
ttl_cache2.put(query_d, answer_d)
print(f"Stored answer with 3s TTL")

# Check freshness at different points
for wait in [0, 2, 4]:
    time.sleep(2 if wait > 0 else 0)
    result, status = ttl_cache2.get(query_d)
    age = round(time.time() - ttl_cache2.cache[query_d.strip().lower()]["timestamp"], 1)
    print(f"  At ~{wait}s: status={status}, age={age}s")


### 11. Comparison of Strategies


In [ ]:
print("\n" + "=" * 60)
print("STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Strategy':<25} {'Trigger':<30} {'Pros':<30} {'Cons':<30}")
print("-" * 115)
print(f"{'TTL':<25} {'Time expiration':<30} {'Simple, predictable':<30} {'May invalidate fresh data':<30}")
print(f"{'Content-Based':<25} {'Source document changes':<30} {'Precise, accurate':<30} {'Requires hash tracking':<30}")
print(f"{'Manual':<25} {'User/admin action':<20} {'Full control':<30} {'Requires human oversight':<30}")


### 12. Combined Strategy Example


In [ ]:
# In production, you often combine strategies. Here is a simple example:
# TTL as a safety net, content-hash as primary, manual as override.

class CombinedCache:
    """Cache combining TTL, content-hash, and manual invalidation."""

    def __init__(self, ttl_seconds=600):
        self.ttl = ttl_seconds
        self.cache = {}
        self.manual_invalidations = set()
        self.content_hash = None

    def _make_key(self, query):
        return query.strip().lower()

    def set_content_hash(self, text):
        self.content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

    def get(self, query, current_hash=None):
        key = self._make_key(query)
        # Check manual invalidation first
        if key in self.manual_invalidations:
            return None, "manually_invalidated"
        if key not in self.cache:
            return None, "miss"
        entry = self.cache[key]
        # Check content hash
        if current_hash and current_hash != entry.get("source_hash"):
            return None, "content_changed"
        # Check TTL
        age = time.time() - entry["timestamp"]
        if age > self.ttl:
            return None, "ttl_expired"
        return entry["answer"], "hit"

    def put(self, query, answer):
        key = self._make_key(query)
        self.cache[key] = {
            "answer": answer,
            "timestamp": time.time(),
            "source_hash": self.content_hash,
        }
        self.manual_invalidations.discard(key)

    def invalidate(self, query):
        self.manual_invalidations.add(self._make_key(query))

    def invalidate_all(self):
        self.cache.clear()
        self.manual_invalidations.clear()

    def size(self):
        return len(self.cache)

combined = CombinedCache(ttl_seconds=300)
combined.set_content_hash(raw_text)

query_e = "What does the White Rabbit say?"
answer_e = generate_answer(query_e)
combined.put(query_e, answer_e)

# Check all strategies
result, status = combined.get(query_e, current_hash=combined.content_hash)
print(f"Combined cache lookup (all valid): status={status}")
result, status = combined.get(query_e, current_hash="different_hash")
print(f"Combined cache lookup (hash changed): status={status}")
combined.invalidate(query_e)
result, status = combined.get(query_e, current_hash=combined.content_hash)
print(f"Combined cache lookup (manual invalidation): status={status}")

print("\nCombined cache size:", combined.size())


### Summary
